# Ray Train 分布式微调练习

本 Notebook 面向大三数据科学专业学生，帮助你练习 **Ray Train** 分布式训练的核心用法。

## 学习目标
- 理解 Ray Train 的 `TorchTrainer` 工作原理
- 掌握 `ScalingConfig`、`RunConfig` 的配置方法
- 学会在训练函数中使用 `train.get_context()` 获取 Worker 信息
- 学会使用 `train.torch.prepare_model()` 和 `train.torch.prepare_data_loader()` 进行分布式包装
- 学会使用 `train.report()` 上报训练指标

## 任务
请搜索所有 `# TODO:` 注释，填写缺失的 Ray Train 核心代码，使训练能够正常运行。


In [ ]:
# ========== 环境检查 & 依赖安装（按需取消注释）==========
# !pip install ray[train] torch transformers datasets

In [2]:
import os

import ray
import torch
from datasets import load_dataset
from ray import train
from ray.train import RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    get_linear_schedule_with_warmup,
)

# ========== 配置本地路径 ==========
MODEL_PATH = "/data/project2/model"
DATASET_PATH = "/data/project2/data"

## 1. 数据预处理函数

将 DataMind-12K 的 trajectory 字段转换为训练序列。这部分代码已完整提供，不需要修改。

In [3]:
def preprocess_dataset(dataset, tokenizer, max_length: int = 2048):
    """
    将 DataMind-12K 的 trajectory 字段转换为 tokenized 训练数据。

    trajectory 结构（每行一条完整对话轨迹）:
      [
        {"role": "system",    "content": "You are an expert..."},
        {"role": "user",      "content": "请分析...数据..."},
        {"role": "assistant", "content": "  thinking...\n  response...\n<code>..."},
        {"role": "user",      "content": "<interpreter>...运行结果...</interpreter>"},
        {"role": "assistant", "content": "  thinking...\n<answer>...</answer>"},
        ...
      ]

    对于 SFT，我们将整个对话序列拼接，并在 labels 中只对 assistant 部分计算 loss。
    通过 apply_chat_template 一次性渲染完整对话文本，再利用 offset mapping 将
    assistant 消息的字符边界精确映射到 token 索引，从而避免 BPE 上下文依赖和
    system-only 消息导致的模板渲染失败问题。
    """

    def tokenize_fn(examples: dict) -> dict:
        """批量 tokenize，对 assistant 消息计算 loss。"""
        input_ids_list = []
        labels_list = []

        assistant_marker = "的助理"
        msg_start_marker = "的"

        for trajectory in examples["trajectory"]:
            messages = [{"role": m["role"], "content": m["content"]}
                        for m in trajectory]

            # 一次性渲染完整对话文本
            full_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )

            # 使用 offset mapping 进行 tokenize，确保字符位置到 token 索引的精确映射
            encoding = tokenizer(
                full_text,
                return_offsets_mapping=True,
                add_special_tokens=False,
            )
            input_ids = encoding["input_ids"]
            offsets = encoding["offset_mapping"]

            # 默认所有位置不参与 loss 计算
            labels = [-100] * len(input_ids)

            # 在文本中查找所有 assistant 消息的起止字符位置
            search_start = 0
            while True:
                start = full_text.find(assistant_marker, search_start)
                if start == -1:
                    break

                # 找到下一条消息的开头（或文本末尾）
                next_msg = full_text.find(msg_start_marker, start + len(assistant_marker))
                if next_msg == -1:
                    end = len(full_text)
                else:
                    end = next_msg

                # 将字符位置映射到 token 索引
                start_token = None
                end_token = None
                for i, (char_s, char_e) in enumerate(offsets):
                    if start_token is None and char_s <= start < char_e:
                        start_token = i
                    if end_token is None and char_s < end <= char_e:
                        end_token = i
                        break
                    if end_token is None and char_s == end:
                        end_token = i - 1
                        break

                if start_token is None:
                    for i, (char_s, char_e) in enumerate(offsets):
                        if char_s <= start < char_e:
                            start_token = i
                if end_token is None:
                    end_token = len(offsets) - 1

                # 只对 assistant 部分的 token 计算 loss
                if start_token is not None and end_token is not None:
                    for i in range(start_token, end_token + 1):
                        labels[i] = input_ids[i]

                search_start = end

            # 截断到 max_length
            if len(input_ids) > max_length:
                input_ids = input_ids[:max_length]
                labels = labels[:max_length]

            input_ids_list.append(input_ids)
            labels_list.append(labels)

        return {"input_ids": input_ids_list, "labels": labels_list}

    return dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=dataset.column_names,
        desc="Tokenizing trajectories",
    )

## 2. 训练函数（需要填写 Ray Train 核心调用）

这是每个 Ray Train Worker 都会执行的训练函数。你需要填写 `# TODO:` 标记的部分。

### 提示
- `train.torch.prepare_model(model)` 将模型包装为 DDP 分布式数据并行
- `train.torch.prepare_data_loader(dataloader)` 将 DataLoader 分片给不同 Worker
- `train.report(metrics=...)` 在每个 epoch 结束后上报训练指标

In [4]:
def train_func(config: dict):
    """
    全量微调的训练函数。

    每个 Worker 独立加载模型和数据集，通过 DDP 同步梯度。

    Args:
        config: 超参数配置字典
    """

    # -------------------------------------------------------------------------
    # 读取配置
    # -------------------------------------------------------------------------
    model_path = config.get("model_path")
    epochs = config.get("epochs", 3)
    batch_size = config.get("batch_size", 2)
    learning_rate = config.get("lr", 2e-4)
    max_length = config.get("max_length", 2048)
    grad_accum = config.get("gradient_accumulation_steps", 8)

    # ★★★ 获取 Ray Train Worker 信息 ★★★
    context = train.get_context()
    worker_rank = context.get_world_rank()
    world_size = context.get_world_size()

    print(f"[Worker {worker_rank}/{world_size}] 开始初始化...")

    # 双重保险：确认路径存在
    assert os.path.exists(model_path), f"Worker 找不到模型路径: {model_path}"

    # -------------------------------------------------------------------------
    # 加载 Tokenizer
    # -------------------------------------------------------------------------
    tokenizer = AutoTokenizer.from_pretrained(
        model_path,
        local_files_only=True,
        padding_side="right",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # -------------------------------------------------------------------------
    # 加载基座模型
    # -------------------------------------------------------------------------
    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        local_files_only=True,
        torch_dtype=dtype,
    )
    # 让模型感知 pad_token
    model.config.pad_token_id = tokenizer.pad_token_id

    # -------------------------------------------------------------------------
    # 全量微调：所有参数均可训练
    # -------------------------------------------------------------------------
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    if worker_rank == 0:
        print(f"可训练参数: {trainable:,} / {total:,} "
              f"({100 * trainable / total:.2f}%)")

    # -------------------------------------------------------------------------
    # 加载并预处理 DataMind-12K 数据集
    # -------------------------------------------------------------------------
    raw_dataset = load_dataset(DATASET_PATH, split="train")
    raw_dataset = raw_dataset.select(range(min(2000, len(raw_dataset))))
    if worker_rank == 0:
        print(f"数据集加载完成，取前 2000 条，共 {len(raw_dataset)} 条样本")

    # Tokenize 并仅对 assistant 消息计算 loss
    tokenized_dataset = preprocess_dataset(raw_dataset, tokenizer, max_length)

    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        return_tensors="pt",
    )

    dataloader = DataLoader(
        tokenized_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collator,
    )

    # ★★★ 请在此处对 DataLoader 进行 Ray Train 分片包装 ★★★
    # TODO: 将 dataloader 包装为 Ray Train 分布式 DataLoader
    dataloader = train.torch.prepare_data_loader(dataloader)

    # ★★★ 请在此处对模型进行 DDP 包装 ★★★
    # TODO: 将 model 包装为 Ray Train DDP 模型
    model = train.torch.prepare_model(model)

    # 优化器 & 学习率调度器
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    total_steps = (len(tokenized_dataset) // (batch_size * world_size)) * epochs
    warmup_steps = max(1, int(total_steps * 0.03))
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    # -------------------------------------------------------------------------
    # 训练循环
    # -------------------------------------------------------------------------
    model.train()
    global_step = 0

    for epoch in range(epochs):
        total_loss = 0.0
        num_batches = 0

        for batch in dataloader:
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss / grad_accum
            loss.backward()

            num_batches += 1

            # 梯度累积
            if (num_batches % grad_accum == 0) or (num_batches == len(dataloader)):
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

            total_loss += loss.item() * grad_accum

        avg_loss = total_loss / num_batches

        # ★★★ 请在此处上报训练指标 ★★★
        # TODO: 调用 train.report() 上报 epoch, loss, learning_rate, global_step
        train.report(metrics={
            "epoch": epoch + 1,
            "loss": avg_loss,
            "learning_rate": scheduler.get_last_lr()[0],
            "global_step": global_step
        })

        if worker_rank == 0:
            print(f"[Epoch {epoch + 1}/{epochs}] Loss: {avg_loss:.4f}, "
                  f"LR: {scheduler.get_last_lr()[0]:.2e}, Step: {global_step}")

    print(f"[Worker {worker_rank}] 训练完成！")

## 3. 启动 Ray Train 分布式训练（需要填写配置代码）

在这一部分，你需要配置 Ray Train 的核心组件并启动训练。

### 提示
- `ScalingConfig(num_workers=..., use_gpu=...)` 配置分布式训练的 Worker 数量和是否使用 GPU
- `RunConfig(name=..., storage_path=...)` 配置实验名称和结果存储路径
- `TorchTrainer(train_loop_per_worker=..., train_loop_config=..., scaling_config=..., run_config=...)` 创建训练器

In [5]:
def main():
    # -------------------------------------------------------------------------
    # 初始化 Ray
    # -------------------------------------------------------------------------
    if not ray.is_initialized():
        context = ray.init()
    else:
        context = ray.get_runtime_context()

    dashboard_url = (
        getattr(context, "dashboard_url", None)
        or getattr(context, "webui_url", "未启用")
    )
    print(f"Ray 已启动，Dashboard: {dashboard_url}")

    # -------------------------------------------------------------------------
    # 训练超参数
    # -------------------------------------------------------------------------
    train_config = {
        "model_path": MODEL_PATH,
        "epochs": 3,
        "batch_size": 2,               # 每条轨迹较长，batch 不宜太大
        "lr": 2e-4,
        "max_length": 2048,
        "gradient_accumulation_steps": 8,
    }

    # -------------------------------------------------------------------------
    # ★★★ 请填写 ScalingConfig ★★★
    # TODO: 创建 ScalingConfig，设置 num_workers=2，use_gpu 根据 CUDA 可用性决定
    scaling_config = ScalingConfig(num_workers= 2, use_gpu= torch.cuda.is_available())

    # -------------------------------------------------------------------------
    # ★★★ 请填写 RunConfig ★★★
    # TODO: 创建 RunConfig，设置 name="qwen-datamind-full" 和 storage_path
    storage_path = os.path.abspath("ray_results")
    os.makedirs(storage_path, exist_ok=True)
    run_config = RunConfig(name= "qwen-datamind-full", storage_path= storage_path)

    # -------------------------------------------------------------------------
    # ★★★ 请填写 TorchTrainer 并启动训练 ★★★
    # TODO: 创建 TorchTrainer 并调用 trainer.fit() 启动训练
    trainer = TorchTrainer(
        train_loop_config=train_config,
        train_loop_per_worker=train_func,
        scaling_config=scaling_config,
        run_config=run_config
    )
    result = trainer.fit()

    # -------------------------------------------------------------------------
    # 训练结果
    # -------------------------------------------------------------------------
    print("\n" + "=" * 60)
    print("微调完成！")
    print("=" * 60)
    print(f"实验路径:   {result.path}")
    print(f"最终指标:   {result.metrics}")
    print(f"Checkpoint: {result.checkpoint}")
    print("=" * 60)

    if ray.is_initialized():
        ray.shutdown()
    print("\nRay 已关闭，演示结束。")

In [ ]:
# ========== 运行训练 ==========
main()

2026-05-21 17:13:54,670	INFO worker.py:1998 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
